In [1]:
import sys
print(sys.executable)

C:\Users\kaush\MyProject\urban-pipeline\venv\Scripts\python.exe


# Taxi Data Ingestion — Step by Step

This notebook walks through `taxi_ingestion.py` cell-by-cell to understand each piece.

**Goal:** Pull NYC taxi trips from BigQuery's public dataset → clean → write parquet → upload to GCS.

In [2]:
# Core Python
import logging
import os
from datetime import datetime, timezone
from pathlib import Path

# Data manipulation
import pandas as pd

# Google Cloud SDKs
from google.cloud import bigquery, storage

print("Imports OK")
print(f"Pandas version: {pd.__version__}")

Imports OK
Pandas version: 2.2.3


## Configuration

In the production script we read these from environment variables. In a notebook we set them directly so you can tweak and re-run cells.

In [23]:
PROJECT_ID = "urban-pipeline-kd-2026"
RAW_BUCKET = "urban-pipeline-kd-2026-raw"

# Start small for the walkthrough — 5,000 rows is plenty to see the mechanics
ROW_LIMIT = 5000

# Source: BigQuery public dataset
PUBLIC_DATASET = "bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2022"

# When did we ingest? Used for partitioning the GCS path
INGESTION_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")
GCS_OBJECT_NAME = f"taxi/notebook_test/ingestion_date={INGESTION_DATE}/taxi_trips_sample.parquet"

# Local temp file (deleted at the end)
LOCAL_TEMP = Path("temp_taxi_sample.parquet")

print(f"Project: {PROJECT_ID}")
print(f"Bucket: {RAW_BUCKET}")
print(f"Pulling {ROW_LIMIT:,} rows from {PUBLIC_DATASET}")
print(f"Will land at: gs://{RAW_BUCKET}/{GCS_OBJECT_NAME}")

Project: urban-pipeline-kd-2026
Bucket: urban-pipeline-kd-2026-raw
Pulling 5,000 rows from bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2022
Will land at: gs://urban-pipeline-kd-2026-raw/taxi/notebook_test/ingestion_date=2026-05-03/taxi_trips_sample.parquet


## Authentication

When we instantiate a BigQuery or Storage client, the SDK looks for credentials in this order:

1. `GOOGLE_APPLICATION_CREDENTIALS` env var pointing to a service account JSON key
2. **Application Default Credentials (ADC)** — the file created by `gcloud auth application-default login`
3. Compute Engine / Cloud Run metadata service (when running on GCP infra)

We have ADC set up, so the SDK picks that up automatically. **No keys, no passwords, no secrets in our code.**

This is the right way to authenticate in dev. In production (Dataproc, Cloud Run, GKE), credential 3 takes over — your workload identity comes from the runtime, not a key file.

In [4]:
bq_client = bigquery.Client(project=PROJECT_ID)
print(f"BQ client connected to project: {bq_client.project}")
print(f"Default location: {bq_client.location}")

BQ client connected to project: urban-pipeline-kd-2026
Default location: None


## The SQL query

We're pulling from `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2022` — a free public dataset Google maintains.

Three things to internalize:

1. **WHERE filter is critical** — without it, BQ scans the entire 2022 table (~90 GB). We filter to January, which is ~7 GB. **You pay per byte scanned.**
2. **LIMIT does NOT save scan cost.** It's applied AFTER the WHERE filter. The full month still gets scanned to find rows; LIMIT only caps how many come back.
3. **Always select specific columns**, never `SELECT *` on huge tables. BigQuery is columnar — fewer columns = fewer bytes scanned = lower cost.

In [24]:
sql = f"""
SELECT
    vendor_id,
    pickup_datetime,
    dropoff_datetime,
    passenger_count,
    trip_distance,
    fare_amount,
    tip_amount,
    total_amount,
    pickup_location_id,
    dropoff_location_id
FROM `{PUBLIC_DATASET}`
WHERE pickup_datetime >= '2022-01-01'
  AND pickup_datetime <  '2022-02-01'
LIMIT {ROW_LIMIT}
"""

print(sql)


SELECT
    vendor_id,
    pickup_datetime,
    dropoff_datetime,
    passenger_count,
    trip_distance,
    fare_amount,
    tip_amount,
    total_amount,
    pickup_location_id,
    dropoff_location_id
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2022`
WHERE pickup_datetime >= '2022-01-01'
  AND pickup_datetime <  '2022-02-01'
LIMIT 5000



In [18]:
# Verify: what does the variable `sql` actually contain right now?
print(sql)


SELECT
    vendor_id,
    pickup_datetime,
    dropoff_datetime,
    passenger_count,
    trip_distance,
    fare_amount,
    tip_amount,
    total_amount,
    pickup_location_id,
    dropoff_location_id
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2022`
WHERE pickup_datetime >= '2022-01-01'
  AND pickup_datetime <  '2022-12-31'
LIMIT 50000



In [25]:
# A "dry run" tells you exactly how many bytes a query would scan,
# WITHOUT actually running it. It costs $0.
# ALWAYS use this on unfamiliar queries before running for real.

dry_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
dry_job = bq_client.query(sql, job_config=dry_config)

bytes_scanned = dry_job.total_bytes_processed
mb_scanned = bytes_scanned / 1024 / 1024
gb_scanned = bytes_scanned / 1024 / 1024 / 1024
estimated_cost = (bytes_scanned / 1024**4) * 5  # $5 per TB scanned (on-demand pricing)

print(f"Bytes scanned (estimate): {bytes_scanned:,}")
print(f"  = {mb_scanned:,.2f} MB")
print(f"  = {gb_scanned:.4f} GB")
print(f"Estimated cost: ${estimated_cost:.4f}")

Bytes scanned (estimate): 3,637,884,291
  = 3,469.36 MB
  = 3.3880 GB
Estimated cost: $0.0165


In [20]:
# Test query 1: narrow date range
narrow_sql = f"""
SELECT vendor_id, pickup_datetime, fare_amount
FROM `{PUBLIC_DATASET}`
WHERE pickup_datetime >= '2022-01-01'
  AND pickup_datetime < '2022-02-01'
"""

# Test query 2: full year
wide_sql = f"""
SELECT vendor_id, pickup_datetime, fare_amount
FROM `{PUBLIC_DATASET}`
WHERE pickup_datetime >= '2022-01-01'
  AND pickup_datetime < '2023-01-01'
"""

dry_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)

narrow_bytes = bq_client.query(narrow_sql, job_config=dry_config).total_bytes_processed
wide_bytes = bq_client.query(wide_sql, job_config=dry_config).total_bytes_processed

print(f"January only:    {narrow_bytes / 1024**3:.2f} GB  (${(narrow_bytes/1024**4)*5:.4f})")
print(f"Full year 2022:  {wide_bytes / 1024**3:.2f} GB  (${(wide_bytes/1024**4)*5:.4f})")
print(f"Ratio:           {wide_bytes / narrow_bytes:.1f}x")

January only:    0.91 GB  ($0.0045)
Full year 2022:  0.91 GB  ($0.0045)
Ratio:           1.0x


In [21]:
# What does BQ know about this table?
table_ref = "bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2022"
table = bq_client.get_table(table_ref)

print(f"Table:               {table.table_id}")
print(f"Total rows:          {table.num_rows:,}")
print(f"Total size:          {table.num_bytes / 1024**3:.2f} GB")
print(f"Time partitioning:   {table.time_partitioning}")
print(f"Clustering fields:   {table.clustering_fields}")
print(f"Created:             {table.created}")

Table:               tlc_yellow_trips_2022
Total rows:          36,256,539
Total size:          6.97 GB
Time partitioning:   None
Clustering fields:   None
Created:             2022-09-14 04:13:35.814000+00:00


In [22]:
# Same date filter, but selecting different columns
one_col_sql = f"""
SELECT vendor_id
FROM `{PUBLIC_DATASET}`
WHERE pickup_datetime >= '2022-01-01' AND pickup_datetime < '2022-02-01'
"""

ten_col_sql = f"""
SELECT
    vendor_id, pickup_datetime, dropoff_datetime,
    passenger_count, trip_distance, fare_amount,
    tip_amount, total_amount,
    pickup_location_id, dropoff_location_id
FROM `{PUBLIC_DATASET}`
WHERE pickup_datetime >= '2022-01-01' AND pickup_datetime < '2022-02-01'
"""

dry_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)

one_col = bq_client.query(one_col_sql, job_config=dry_config).total_bytes_processed
ten_col = bq_client.query(ten_col_sql, job_config=dry_config).total_bytes_processed

print(f"1 column selected:   {one_col / 1024**3:.2f} GB  (${(one_col/1024**4)*5:.4f})")
print(f"10 columns selected: {ten_col / 1024**3:.2f} GB  (${(ten_col/1024**4)*5:.4f})")
print(f"Ratio:               {ten_col / one_col:.1f}x")

1 column selected:   0.37 GB  ($0.0018)
10 columns selected: 3.39 GB  ($0.0165)
Ratio:               9.1x
